# Deploy TravelMind End to End

Take the working local agent to a managed **AgentCore Runtime** endpoint others
can call. This notebook runs the local steps for real, **offline** (a
deterministic mock stands in for the model, so no credentials are needed), and
lays out the cloud steps as the exact commands you run in your own terminal.

Pair this with `deploy_runbook.md`. Files used: `travelmind_agent.py`,
`travelmind_runtime.py`, `Dockerfile`, `iam_invoke_policy.json`.

**The seven moves:** local agent -> wrap -> test the contract -> containerize ->
`configure` -> `launch` -> `invoke`, then clear the 404 and the 403.

## Setup

**VS Code (3 steps)**
```bash
python -m venv .venv && source .venv/bin/activate     # 1. activate venv
aws configure                                         # 2. creds + region us-east-1
pip install -r requirements.txt                       # 3. install deps
```

**Google Colab (3 steps)**
```python
!pip install -q strands-agents bedrock-agentcore boto3   # 1. install
import os                                                  # 2. creds via Colab Secrets
os.environ["AWS_ACCESS_KEY_ID"] = "..."
os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"            # 3. region
```

**LIVE flag:** the next cell defaults to offline. Everything runs with the mock.
Flip `LIVE = True` once you have credentials to use the real model.

In [1]:
# ---- config ----
LIVE = False   # False: offline mock, no AWS. True: real model via Bedrock.
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
REGION = "us-east-1"

import os
os.environ["LIVE"] = "1" if LIVE else "0"   # travelmind_runtime reads this
print("LIVE =", LIVE, "| model =", MODEL_ID, "| region =", REGION)

LIVE = False | model = us.anthropic.claude-haiku-4-5-20251001-v1:0 | region = us-east-1


## Step 1. Confirm the local agent

The three tools are plain Python with no AWS dependency, so they run offline.
`build_agent()` (the real model) is shown but only called when `LIVE` is True.

In [2]:
from travelmind_agent import lookup_booking, get_rebooking_options, mock_agent, get_agent

# tools run with no credentials
print("lookup :", lookup_booking("JX48Q2"))
print("options:", get_rebooking_options("JX48Q2"))

# the agent: deterministic mock offline, real Bedrock model when LIVE
agent = (lambda t: str(get_agent()(t))) if LIVE else mock_agent
print("\nagent  :", agent("My flight on JX48Q2 was cancelled. Options?"))

lookup : {'pnr': 'JX48Q2', 'status': 'CANCELLED', 'flight': 'AI-302'}
options: [{'flight': 'AI-318', 'dep': '18:40'}, {'flight': '6E-552', 'dep': '21:15'}]

agent  : PNR JX48Q2 is cancelled due to weather. Rebooking options: AI-318 at 18:40, 6E-552 at 21:15.


## Step 2. Wrap it: the entrypoint and payload contract

`travelmind_runtime.py` adds exactly three things to the working agent. Your
agent logic is untouched.

```python
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from travelmind_agent import get_agent

app = BedrockAgentCoreApp()                 # 1. the runtime app object

@app.entrypoint                              # 2. the function Runtime calls per request
def invoke(payload):
    return str(get_agent()(payload.get("prompt", "")))

if __name__ == "__main__":
    app.run()                                # 3. serves /invocations + /ping on :8080
```

Callers now send `{"prompt": "..."}` and get a string back. That fixed shape is
what makes it an endpoint, not a function.

In [3]:
# The entrypoint is callable in-process. This is EXACTLY what AgentCore runs
# per request, tested here with no server and no AWS (LIVE=False -> mock).
import importlib, travelmind_runtime
importlib.reload(travelmind_runtime)        # pick up the LIVE env set above
from travelmind_runtime import invoke

print(invoke({"prompt": "Status of PNR JX48Q2?"}))
print(invoke({"prompt": "Rebook JX48Q2"}))

PNR JX48Q2 is cancelled due to weather. Rebooking options: AI-318 at 18:40, 6E-552 at 21:15.
PNR JX48Q2 is cancelled due to weather. Rebooking options: AI-318 at 18:40, 6E-552 at 21:15.


## Step 3. Run and test the contract locally

Locally, start the server (it serves `/ping` and `/invocations` on 8080):
```bash
python travelmind_runtime.py
```
`app.run()` blocks, so test from a **second terminal**:
```bash
curl localhost:8080/ping
# {"status": "healthy"}

curl -X POST localhost:8080/invocations \
     -H 'Content-Type: application/json' \
     -d '{"prompt":"Status of PNR JX48Q2?"}'
```
In a notebook, the in-process `invoke(...)` call above is the same contract.

**Gate:** if it fails here, it fails in the cloud too. Never deploy a red contract.

## Step 4. Containerize

The two lines in the `Dockerfile` that matter for the runtime are `EXPOSE 8080`
and the `CMD` that starts the server. The starter toolkit can build this for you;
it is written out so you can see what it does.

In [4]:
print(open("Dockerfile").read())

# Dockerfile  -  packages the TravelMind agent for AgentCore Runtime.
# The starter toolkit (agentcore configure/launch) can generate and build this
# for you. It is written out in full here so you can see exactly what it does.

# slim base: small image, short cold start.
FROM python:3.12-slim
WORKDIR /app

# Copy requirements FIRST and install, so Docker caches the (slow) pip layer.
# Changing your code then does not re-run pip.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Now copy the application code.
COPY . .

# The runtime contract: the container must listen on 8080.
EXPOSE 8080

# Start the server. travelmind_runtime.py calls app.run(), which serves
# POST /invocations and GET /ping on 8080.
CMD ["python", "travelmind_runtime.py"]

# What changes in production:
#   - Pin a digest, not just a tag, for the base image (reproducible builds).
#   - Run as a non-root user (add: RUN useradd app && USER app).
#   - Do not bake secrets into the image. Pass

## Step 5. `agentcore configure`

```bash
agentcore configure --entrypoint travelmind_runtime.py
```
Creates `.bedrock_agentcore.yaml`, an **execution role** (the identity the running
agent uses), and an **ECR** target for the image.

Know which CLI you have: the **starter toolkit** (`configure / launch / invoke`,
Python only), not the `@aws/agentcore` npm CLI (`create / deploy`). If `launch` is
missing but `deploy` exists, you are on the other CLI.

## Step 6. `agentcore launch`

```bash
agentcore launch
```
In order: build the image, push to ECR, provision the managed runtime (network
interfaces, scaling, health checks), return a **runtime ARN**.

About 5 to 10 minutes. That is one-time provisioning, not per-call latency.

## Step 7. `agentcore invoke`

The simplest path from your terminal:
```bash
agentcore invoke '{"prompt":"My flight on JX48Q2 was cancelled. Options?"}'
```
Or call the deployed endpoint from Python (runs only when `LIVE` and you paste the
runtime ARN from `launch`):

In [5]:
# Calls the DEPLOYED endpoint. Offline this just prints the equivalent command.
RUNTIME_ARN = ""   # paste the agentRuntimeArn from `agentcore launch`

if LIVE and RUNTIME_ARN:
    import boto3, json
    rt = boto3.client("bedrock-agentcore", region_name=REGION)
    resp = rt.invoke_agent_runtime(
        agentRuntimeArn=RUNTIME_ARN,
        payload=json.dumps({"prompt": "Status of PNR JX48Q2?"}).encode(),
    )
    print(resp["response"].read().decode())   # streamed response body
else:
    print("offline. To call the deployed agent: set LIVE=True and paste RUNTIME_ARN,")
    print("or from a terminal run:  agentcore invoke '{\"prompt\":\"Status of JX48Q2?\"}'")

offline. To call the deployed agent: set LIVE=True and paste RUNTIME_ARN,
or from a terminal run:  agentcore invoke '{"prompt":"Status of JX48Q2?"}'


## The wall: the two errors almost everyone hits

| Code | Symptom | Cause | Fix |
|---|---|---|---|
| 404 | model ARN not found | model id missing the `us.` prefix | use the inference profile `us.anthropic.claude-haiku-4-5-20251001-v1:0` |
| 403 | access denied on invoke | execution role cannot invoke that model | attach the policy below (replace `ACCOUNT_ID`) |

Fix the 404 first (right model id), then the 403 (right permissions on that id).
The profile is multi-region, so the role needs the inference-profile ARN **and**
the three regional foundation-model ARNs it fronts.

In [6]:
print(open("iam_invoke_policy.json").read())

{
  "_comment": "Attach this to the agent's EXECUTION ROLE to clear the 403 on invoke. It grants invoke on the inference profile AND on the three regional foundation-model ARNs the profile fronts. Replace ACCOUNT_ID. This is for Haiku 4.5; add a second statement (or more ARNs) if you also use Sonnet.",
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "InvokeHaikuViaInferenceProfile",
      "Effect": "Allow",
      "Action": [
        "bedrock:InvokeModel",
        "bedrock:InvokeModelWithResponseStream"
      ],
      "Resource": [
        "arn:aws:bedrock:us-east-1:ACCOUNT_ID:inference-profile/us.anthropic.claude-haiku-4-5-20251001-v1:0",
        "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-haiku-4-5-20251001-v1:0",
        "arn:aws:bedrock:us-east-2::foundation-model/anthropic.claude-haiku-4-5-20251001-v1:0",
        "arn:aws:bedrock:us-west-2::foundation-model/anthropic.claude-haiku-4-5-20251001-v1:0"
      ]
    },
    {
      "Sid": "WriteAgentLogs",

## What changes in production

| In this demo | In production |
|---|---|
| `aws configure` keys | the execution role, no long-lived keys in code or image |
| hardcoded region | injected via environment / config |
| broad-ish role | least privilege: invoke only the models you use, write only your log group |
| no retries | adaptive retries + a request timeout; throttles are normal at scale |
| logs only | observability on at deploy (CloudWatch GenAI spans) |

## Cleanup

```bash
agentcore destroy        # tear down the runtime this config created
```
Also remove the ECR image and log group if you will not reuse them.

**Recap:** the same agent now answers from a managed, isolated, scalable
endpoint. That is stages 1 to 4 of the pipeline. Next: version it and ship safely
(`release_pipeline.md`, `release.py`).